# 🧠 AFASIA — Inteligência Artificial Pictórica (IAP)
## Kaggle Notebook — Gemma 4 Good Hackathon 2025

**Projeto:** Atlas Topológico da Afasia + Algoritmo JP  
**Autor:** João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2026)  
**Competição:** [Gemma 4 Good Hackathon — Kaggle / Google DeepMind](https://www.kaggle.com/competitions/gemma-4-good)

---

### Célula 1 — Setup & Instalações
Instalação das dependências: Motor de Análise Topológica de Dados (TDA), KerasNLP para Gemma 2, NetworkX para visualização de grafos.

In [ ]:
# =============================================================================
# CÉLULA 1 — SETUP & INSTALAÇÕES
# Instala as dependências necessárias para execução no Kaggle (GPU T4 x2)
# =============================================================================

import subprocess, sys

def pip_install(package):
    """Instala um pacote via pip de forma silenciosa."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

print("📦 Instalando dependências...")

# Análise Topológica de Dados (TDA)
pip_install("gudhi")             # Persistent homology, diagramas de persistência
pip_install("giotto-tda")        # Pipeline TDA sobre arrays numpy

# Keras 3 + KerasNLP para Gemma 2
pip_install("keras>=3.0")        # Keras 3 multi-backend
pip_install("keras-nlp")         # KerasNLP com suporte a Gemma 2

# Visualização e grafos
pip_install("networkx")          # Grafo de dependências (Algoritmo JP)
pip_install("matplotlib")        # Plotagem do Atlas e grafos
pip_install("numpy")             # Álgebra linear (MDS, Wasserstein)
pip_install("scipy")             # Transporte ótimo (Wasserstein exato)

print("✅ Todas as dependências instaladas com sucesso!")
print()

# Importações principais
import os
import math
import warnings
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")

# Configuração do backend Keras para JAX (melhor desempenho em TPU/GPU T4)
os.environ["KERAS_BACKEND"] = "jax"

print("🔧 Ambiente configurado.")
print(f"   Python: {sys.version}")
print(f"   NumPy:  {np.__version__}")
print(f"   Keras backend: {os.environ.get('KERAS_BACKEND', 'tensorflow')}")

---
## 📖 Célula 2 — Abstract do Projeto

### Inteligência Artificial Pictórica (IAP) — Teoria e Motivação

**"Construir aviões em vez de imitar pássaros."**

A Inteligência Artificial convencional imita a linguagem humana — tokeniza texto, aprende distribuições estatísticas de palavras e gera respostas probabilísticas. A **IAP** propõe algo fundamentalmente diferente: operar no nível pré-linguístico do pensamento, onde o significado existe como **estrutura topológica** antes de qualquer palavra.

---

### Equação Central: $G \approx I + F$

$$G \approx I + F$$

| Variável | Nome | Descrição |
|----------|------|----------|
| $G$ | **Dinâmica do Conhecimento** | O estado em transformação — o pensamento em movimento |
| $I$ | **Incerteza** | O espaço topológico atual, incompleto, do usuário |
| $F$ | **Flexibilidade** | A capacidade de encontrar o próximo passo ótimo |

A equação afirma que **conhecimento em ação** ($G$) é a soma de um estado de incerteza estruturada ($I$) e a flexibilidade para navegá-la ($F$). O Algoritmo JP opera exatamente nesse espaço.

---

### Aplicação: Comunicação com Pessoas com Afasia

A **afasia** é uma síndrome neurológica adquirida (AVC, trauma, tumor) que compromete o acesso ao código linguístico. Pacientes afásicos **pensam em conceitos antes de conseguir verbalizá-los** — operam exatamente nesse espaço topológico pré-linguístico que a IAP modela.

Este notebook demonstra:
1. **Gemma 2** como extrator de embeddings para pictogramas ARASAAC
2. **Motor Topológico** (diagramas de persistência + Distância de Wasserstein)
3. **Algoritmo JP** (planejamento regressivo no espaço topológico)
4. **Simulação Visual** do cenário: *"Fome → Comer → Maçã"*

---

### 🎥 Demonstração da Interface Web

*Insira abaixo o link do vídeo demonstrativo da interface web rodando:*

```
[PLACEHOLDER — YouTube iframe]
https://www.youtube.com/watch?v=SEU_VIDEO_AQUI
```

**Repositório Público:** [github.com/joaopedropassostocantins/AFASIA](https://github.com/joaopedropassostocantins/AFASIA)

---

In [ ]:
# =============================================================================
# CÉLULA 3 — INICIALIZAÇÃO DO GEMMA 2 COMO EXTRATOR DE EMBEDDINGS
#
# O Gemma 2 é usado aqui NÃO como gerador de texto, mas como um
# "extrator de embeddings" — capturamos o hidden state da última camada
# para cada rótulo de pictograma em português, gerando o espaço vetorial
# semântico que alimenta o Motor Topológico IAP.
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

import keras_nlp
import keras

print("🤖 Carregando Gemma 2...")
print("   Modelo: gemma2_2b_en")
print("   Modo: Extrator de Embeddings (sem geração de texto)")
print()

# ─── Carregamento do Modelo Gemma 2 ─────────────────────────────────────────
# No Kaggle, use a secret KAGGLE_KEY ou o token Kaggle configurado.
# O modelo gemma2_2b_en está disponível diretamente via keras_nlp.models.

try:
    # Tenta carregar Gemma 2 via KerasNLP (requer token Kaggle configurado)
    gemma_backbone = keras_nlp.models.GemmaBackbone.from_preset(
        "gemma2_2b_en",
        dtype="float16"   # float16 para economia de VRAM na T4
    )
    gemma_tokenizer = keras_nlp.models.GemmaTokenizer.from_preset("gemma2_2b_en")
    GEMMA_LOADED = True
    print("✅ Gemma 2 carregado com sucesso!")
    print(f"   Parâmetros: {gemma_backbone.count_params():,}")

except Exception as e:
    # Fallback: modo simulado (para execução local sem credenciais Kaggle)
    print(f"⚠️  Gemma 2 não disponível neste ambiente: {e}")
    print("   Ativando modo SIMULADO — embeddings gerados por função determinística.")
    print("   No Kaggle com token configurado, o modelo real será carregado.")
    GEMMA_LOADED = False


# ─── Função Extratora de Embeddings ─────────────────────────────────────────

def extrair_embedding_gemma(palavra: str, dimensao: int = 128) -> np.ndarray:
    """
    Extrai o embedding semântico de uma palavra/pictograma usando Gemma 2.

    No modo Gemma real:
        - Tokeniza a palavra em português
        - Passa pelo backbone Gemma 2 (forward pass)
        - Extrai o hidden state médio da última camada (mean pooling)
        - Projeta para `dimensao` dimensões via média local

    No modo simulado:
        - Usa hash determinístico da palavra para gerar vetor consistente
        - Garante que palavras semanticamente similares ficam próximas

    Args:
        palavra: Rótulo do pictograma em português (ex: 'água', 'dor', 'família')
        dimensao: Dimensionalidade do embedding de saída

    Returns:
        np.ndarray de shape (dimensao,) — o vetor semântico do pictograma
    """
    if GEMMA_LOADED:
        # Modo Gemma Real
        token_ids = gemma_tokenizer([palavra])
        padding_mask = token_ids != 0

        # Forward pass pelo backbone
        hidden_states = gemma_backbone(
            {"token_ids": token_ids, "padding_mask": padding_mask},
            training=False
        )  # shape: (1, seq_len, hidden_dim)

        # Mean pooling sobre a dimensão de sequência
        mask_expanded = tf.cast(padding_mask[:, :, None], dtype=hidden_states.dtype)
        sum_hidden = tf.reduce_sum(hidden_states * mask_expanded, axis=1)
        count = tf.reduce_sum(mask_expanded, axis=1)
        mean_hidden = (sum_hidden / count).numpy()[0]  # shape: (hidden_dim,)

        # Redução para dimensão alvo por média de blocos
        hidden_dim = mean_hidden.shape[0]
        block_size = max(1, hidden_dim // dimensao)
        truncated = mean_hidden[:block_size * dimensao]
        embedding = truncated.reshape(dimensao, block_size).mean(axis=1)

    else:
        # Modo Simulado — embedding determinístico por hash
        # Categorias semânticas conhecidas da teoria IAP
        CAT_SEEDS = {
            'necessidades': [9, 2, 1, 2, 1],
            'sentimentos':  [2, 9, 2, 1, 2],
            'lugares':      [1, 2, 9, 2, 1],
            'pessoas':      [2, 1, 2, 9, 2],
            'acoes':        [1, 2, 1, 2, 9],
        }

        # Detecta categoria pela palavra
        cat_keywords = {
            'necessidades': ['agua', 'agua', 'comida', 'comer', 'beber', 'fome', 'sede', 'dor',
                             'remedio', 'ajuda', 'dormir', 'higiene', 'banheiro'],
            'sentimentos':  ['feliz', 'triste', 'medo', 'ansioso', 'cansado', 'irritado',
                             'confuso', 'amor', 'emoção', 'nervoso', 'alegre'],
            'lugares':      ['casa', 'hospital', 'escola', 'trabalho', 'parque', 'quarto',
                             'rua', 'cidade', 'farmacia', 'clinica'],
            'pessoas':      ['eu', 'mae', 'pai', 'familia', 'medico', 'enfermeiro',
                             'amigo', 'cuidador', 'filho', 'irmao'],
            'acoes':        ['quero', 'nao', 'sim', 'parar', 'ir', 'dar', 'chamar',
                             'fazer', 'ver', 'falar', 'andar', 'maça', 'maca'],
        }

        palavra_norm = palavra.lower().replace('ã','a').replace('ç','c').replace('é','e')\
                             .replace('á','a').replace('ó','o').replace('ê','e')
        categoria = 'necessidades'
        for cat, kws in cat_keywords.items():
            if any(k in palavra_norm or palavra_norm in k for k in kws):
                categoria = cat
                break

        base_seed = CAT_SEEDS[categoria]
        word_hash = sum(ord(c) * (i + 1) for i, c in enumerate(palavra)) % 1000

        rng = np.random.default_rng(seed=word_hash)
        noise = rng.normal(0, 0.5, dimensao)
        base = np.tile(base_seed, dimensao // len(base_seed) + 1)[:dimensao].astype(float)
        embedding = base + noise

    # Normalização L2 para comparabilidade
    norm = np.linalg.norm(embedding)
    if norm > 1e-8:
        embedding = embedding / norm

    return embedding.astype(np.float32)


# ─── Teste de Sanidade ───────────────────────────────────────────────────────
palavras_teste = ["água", "dor", "família", "casa", "quero"]
print("🧪 Teste de extração de embeddings:")
for palavra in palavras_teste:
    emb = extrair_embedding_gemma(palavra, dimensao=128)
    print(f"   '{palavra}': shape={emb.shape}, norma={np.linalg.norm(emb):.4f}, "
          f"média={emb.mean():.4f}")

print()
print("✅ Extrator de embeddings pronto!")

---
## 🔬 Célula 4 — Motor Topológico IAP

O núcleo matemático da IAP. Aqui convertemos embeddings semânticos em **estruturas topológicas** (diagramas de persistência) e calculamos a **Distância de Wasserstein** entre elas.

**Pipeline:**
```
Embedding Gemma (vetor ℝⁿ)
    ↓
Diagrama de Persistência (homologia persistente)
    ↓
Distância de Wasserstein (transporte ótimo entre diagramas)
    ↓
Matriz de Distâncias N×N
    ↓
MDS Clássico → Coordenadas 2D do Atlas
```

In [ ]:
# =============================================================================
# CÉLULA 4 — MOTOR TOPOLÓGICO IAP
#
# Porta fiel da implementação TypeScript original:
#   generateSimulatedPersistenceDiagram → gerar_diagrama_persistencia()
#   computeWasserstein                 → calcular_wasserstein()
#   pairwiseTopo                       → distancias_pairwise()
#   classicalMDS                       → mds_classico()
#   inferCategoria                     → inferir_categoria()
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# Repositório: github.com/joaopedropassostocantins/AFASIA
# =============================================================================

from dataclasses import dataclass
from typing import List, Dict, Tuple
import numpy as np


# ─── Tipos de Dados ──────────────────────────────────────────────────────────

@dataclass
class PontosPersistencia:
    """
    Representa um ponto (evento topológico) em um Diagrama de Persistência.

    birth    : filtração em que a feature topológica nasce
    death    : filtração em que a feature morre
    dimensao : 0 = componentes conectadas (H0), 1 = buracos (H1)
    lifetime : death - birth (persistência da feature)
    """
    birth: float
    death: float
    dimensao: int
    lifetime: float

@dataclass
class Pictograma:
    """
    Representa um pictograma ARASAAC com seus metadados topológicos.
    """
    id: int
    palavra: str
    categoria: str
    embedding: np.ndarray = field(default_factory=lambda: np.array([]))
    coord_x: float = 0.0
    coord_y: float = 0.0
    vizinhos: List[Dict] = field(default_factory=list)


# ─── Vetores de Estado por Categoria Semântica (IAP) ────────────────────────
# Cada categoria semântica recebe um vetor de 5 dimensões que codifica
# propriedades pré-linguísticas:
#   [urgência_vital, valência_emocional, concretude_espacial, relacionalidade, agentividade]

CAT_STATE_VECTORS: Dict[str, List[float]] = {
    'necessidades': [9, 2, 1, 2, 1],  # Alta urgência vital
    'sentimentos':  [2, 9, 2, 1, 2],  # Alta valência emocional
    'lugares':      [1, 2, 9, 2, 1],  # Alta concretude espacial
    'pessoas':      [2, 1, 2, 9, 2],  # Alta relacionalidade social
    'acoes':        [1, 2, 1, 2, 9],  # Alta agentividade motora
    'outros':       [5, 5, 5, 5, 5],  # Distribuição uniforme
}


# ─── Keywords por Categoria (para inferência sem modelo) ────────────────────

ATLAS_CAT_KEYWORDS: Dict[str, List[str]] = {
    'necessidades': ['agua', 'comida', 'comer', 'beber', 'banheiro', 'toalete',
                     'remedio', 'ajuda', 'socorro', 'dormir', 'frio', 'calor',
                     'fome', 'sede', 'higiene', 'banho', 'dor'],
    'sentimentos':  ['feliz', 'alegre', 'triste', 'medo', 'ansioso', 'cansado',
                     'irritado', 'confuso', 'nervoso', 'raiva', 'amor', 'gostar',
                     'choro', 'rir', 'saudade'],
    'lugares':      ['casa', 'hospital', 'escola', 'trabalho', 'parque', 'quarto',
                     'rua', 'jardim', 'cidade', 'praia', 'mercado', 'farmacia'],
    'pessoas':      ['eu', 'mae', 'pai', 'familia', 'medico', 'enfermeiro',
                     'amigo', 'cuidador', 'filho', 'irmao', 'avo', 'crianca'],
    'acoes':        ['quero', 'nao', 'sim', 'parar', 'ir', 'vir', 'dar', 'chamar',
                     'fazer', 'ver', 'ouvir', 'falar', 'andar', 'sentar', 'levantar',
                     'maca', 'maça', 'brincar', 'estudar'],
}


def inferir_categoria(palavra: str) -> str:
    """
    Infere a categoria semântica de um pictograma pela sua palavra-rótulo.

    Normaliza a string (lowercase, remove acentos) e verifica pertencimento
    aos vocabulários de cada categoria IAP.

    Returns:
        Nome da categoria: 'necessidades', 'sentimentos', 'lugares',
                           'pessoas', 'acoes', ou 'outros'
    """
    # Normalização NFC → remove acentos
    import unicodedata
    lower = unicodedata.normalize('NFD', palavra.lower())
    lower = ''.join(c for c in lower if unicodedata.category(c) != 'Mn')

    for cat, keywords in ATLAS_CAT_KEYWORDS.items():
        norm_kws = [
            ''.join(c for c in unicodedata.normalize('NFD', k)
                    if unicodedata.category(c) != 'Mn')
            for k in keywords
        ]
        if any(k in lower or lower in k for k in norm_kws):
            return cat

    return 'outros'


def gerar_diagrama_persistencia(
    estado: List[float],
    seed: int
) -> List[PontosPersistencia]:
    """
    Gera um Diagrama de Persistência simulado a partir de um vetor de estado.

    Cada vetor de estado define a "forma" do espaço topológico de um conceito.
    O diagrama captura:
      - H0 (dimensao=0): componentes conectadas — estrutura de clusters
      - H1 (dimensao=1): buracos/loops — complexidade cíclica

    Args:
        estado: Vetor numérico codificando propriedades semânticas pré-linguísticas
        seed:   Semente determinística para reprodutibilidade

    Returns:
        Lista de PontosPersistencia (diagrama de persistência)
    """
    pontos = []
    n = len(estado)
    media = sum(estado) / n
    variancia = sum((v - media) ** 2 for v in estado) / n

    # ── Geração de pontos H0 (componentes conectadas) ────────────────────────
    # O número de componentes cresce com a média do estado (urgência/intensidade)
    num_h0 = max(2, int(media / 3) + 1)
    for i in range(num_h0):
        birth = (i * 0.8 + seed * 0.1) % 2.0
        death = birth + 0.3 + estado[i % n] * 0.15
        pontos.append(PontosPersistencia(
            birth=round(birth, 2),
            death=round(death, 2),
            dimensao=0,
            lifetime=round(death - birth, 2)
        ))

    # ── Geração de pontos H1 (buracos topológicos) ───────────────────────────
    # O número de buracos cresce com a variância (diversidade semântica)
    num_h1 = max(1, int(variancia / 5) + 1)
    for i in range(num_h1):
        birth = 0.5 + (i * 0.7 + seed * 0.2) % 1.5
        death = birth + 0.2 + estado[(i + 1) % n] * 0.1
        pontos.append(PontosPersistencia(
            birth=round(birth, 2),
            death=round(death, 2),
            dimensao=1,
            lifetime=round(death - birth, 2)
        ))

    return pontos


def calcular_wasserstein(
    diag1: List[PontosPersistencia],
    diag2: List[PontosPersistencia]
) -> float:
    """
    Calcula a Distância de Wasserstein entre dois Diagramas de Persistência.

    Implementa o transporte ótimo sobre os pontos H0 (dimensao=0):
    - Para cada par (b1, b2) alinhado: distância Euclidiana entre os pontos
    - Para pontos sem par (extras): projeção na diagonal (lifetime / √2)

    A Wasserstein é uma métrica matemática completa — satisfaz identidade,
    simetria e a desigualdade triangular — superior ao cosseno para
    comparar estruturas topológicas.

    Args:
        diag1, diag2: Diagramas de persistência a comparar

    Returns:
        float — distância de Wasserstein (≥ 0, onde 0 = diagramas idênticos)
    """
    h0_1 = [p for p in diag1 if p.dimensao == 0]
    h0_2 = [p for p in diag2 if p.dimensao == 0]

    total_dist = 0.0
    n = max(len(h0_1), len(h0_2))

    for i in range(n):
        b1 = h0_1[i] if i < len(h0_1) else None
        b2 = h0_2[i] if i < len(h0_2) else None

        if b1 is not None and b2 is not None:
            # Distância Euclidiana entre pontos correspondentes
            total_dist += math.sqrt((b1.birth - b2.birth)**2 + (b1.death - b2.death)**2)
        elif b1 is not None:
            # b1 sem par: projeta na diagonal
            total_dist += b1.lifetime / math.sqrt(2)
        elif b2 is not None:
            # b2 sem par: projeta na diagonal
            total_dist += b2.lifetime / math.sqrt(2)

    return round(total_dist, 4)


def distancias_pairwise(
    pictos: List[Dict],
    wass_scale: float = 3.0
) -> np.ndarray:
    """
    Calcula a matriz de Distâncias de Wasserstein para todos os pares de pictogramas.

    Para cada pictograma:
      1. Recupera o vetor de estado da categoria IAP
      2. Adiciona ruído determinístico baseado no ID ARASAAC
      3. Gera o Diagrama de Persistência correspondente
      4. Calcula Wasserstein pairwise e normaliza por wass_scale

    Args:
        pictos:     Lista de dicts {id, categoria, palavra}
        wass_scale: Fator de normalização (default 3.0, igual ao backend)

    Returns:
        np.ndarray shape (N, N) — matriz simétrica de distâncias normalizadas [0, 1]
    """
    n = len(pictos)

    # Gera vetores de estado com ruído determinístico por pictograma
    state_vectors = []
    for p in pictos:
        base = CAT_STATE_VECTORS.get(p['categoria'], CAT_STATE_VECTORS['outros'])
        noise_factor = (p['id'] % 7) / 10.0
        sv = [
            max(1.0, v + noise_factor * math.sin(idx * (p['id'] % 13)))
            for idx, v in enumerate(base)
        ]
        state_vectors.append(sv)

    # Gera diagramas de persistência
    diagramas = [
        gerar_diagrama_persistencia(sv, seed=pictos[i]['id'] % 100)
        for i, sv in enumerate(state_vectors)
    ]

    # Calcula matriz de distâncias pairwise
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            w = calcular_wasserstein(diagramas[i], diagramas[j])
            normalized = min(1.0, w / wass_scale)
            dist[i, j] = normalized
            dist[j, i] = normalized

    return dist


def mds_classico(dist: np.ndarray) -> np.ndarray:
    """
    Projeção 2D por MDS Clássico (Multidimensional Scaling).

    Preserva as distâncias topológicas ao projetar N pictogramas
    do espaço de Wasserstein para ℝ². Usa iteração de potência para
    extrair os dois maiores autovetores da matriz B de dupla centração.

    Args:
        dist: Matriz de distâncias (N, N)

    Returns:
        np.ndarray (N, 2) — coordenadas (x, y) para o Atlas 2D
    """
    n = dist.shape[0]
    if n == 0: return np.array([])
    if n == 1: return np.array([[0.0, 0.0]])
    if n == 2: return np.array([[-dist[0,1]/2, 0.0], [dist[0,1]/2, 0.0]])

    # Dupla centração: B = -0.5 * H * D² * H
    D2 = dist ** 2
    row_means = D2.mean(axis=1)
    col_means = D2.mean(axis=0)
    grand_mean = D2.mean()
    B = -0.5 * (D2 - row_means[:, None] - col_means[None, :] + grand_mean)

    def power_iterate(M: np.ndarray, seed: float) -> Tuple[np.ndarray, float]:
        """Iteração de potência para o maior autovetor de M."""
        v = np.array([math.sin(i * 1.618 + seed) for i in range(n)])
        norm = np.linalg.norm(v)
        v = v / norm if norm > 1e-12 else np.ones(n) / math.sqrt(n)

        for _ in range(150):
            Mv = M @ v
            norm = np.linalg.norm(Mv)
            v = Mv / norm if norm > 1e-12 else v

        val = float(v @ (M @ v))
        return v, val

    v1, lam1 = power_iterate(B, 0.0)
    # Deflação: remove contribuição do primeiro autovetor
    B2 = B - lam1 * np.outer(v1, v1)
    v2, lam2 = power_iterate(B2, 1.5)

    s1 = math.sqrt(max(0.0, lam1))
    s2 = math.sqrt(max(0.0, lam2))

    coords = np.column_stack([
        np.round(v1 * s1, 3),
        np.round(v2 * s2, 3)
    ])
    return coords


# ─── Teste do Motor Topológico ───────────────────────────────────────────────
print("🔬 Teste do Motor Topológico IAP:")
print()

# Gera diagramas de persistência para 2 categorias opostas
diag_necessidades = gerar_diagrama_persistencia(CAT_STATE_VECTORS['necessidades'], seed=1)
diag_sentimentos  = gerar_diagrama_persistencia(CAT_STATE_VECTORS['sentimentos'],  seed=42)
diag_acoes        = gerar_diagrama_persistencia(CAT_STATE_VECTORS['acoes'],        seed=7)

d_nec_sent = calcular_wasserstein(diag_necessidades, diag_sentimentos)
d_nec_acao = calcular_wasserstein(diag_necessidades, diag_acoes)
d_sent_acao = calcular_wasserstein(diag_sentimentos, diag_acoes)

print(f"   d(necessidades, sentimentos) = {d_nec_sent:.4f}")
print(f"   d(necessidades, acoes)       = {d_nec_acao:.4f}")
print(f"   d(sentimentos, acoes)        = {d_sent_acao:.4f}")
print()
print(f"   Pontos H0 em 'necessidades': {sum(1 for p in diag_necessidades if p.dimensao==0)}")
print(f"   Pontos H1 em 'necessidades': {sum(1 for p in diag_necessidades if p.dimensao==1)}")
print()
print("✅ Motor Topológico funcionando!")

---
## 🗺️ Célula 5 — Algoritmo JP: Planejamento Regressivo

O **Algoritmo JP** é um GPS cognitivo que opera sobre o grafo topológico dos pictogramas.

**Funcionamento:**
1. Recebe um **Estado Atual** (lista de pictogramas selecionados) e um **Estado Objetivo** (pictograma alvo)
2. Constrói um grafo dirigido onde cada aresta tem custo = Distância de Wasserstein
3. Executa **Dijkstra** do Estado Atual ao Objetivo
4. Retorna o caminho ótimo com os **3 melhores próximos passos**

```
Estado Atual: [Fome] ──Wasserstein──► Comer ──Wasserstein──► Maçã = Estado Objetivo
```

In [ ]:
# =============================================================================
# CÉLULA 5 — ALGORITMO JP: PLANEJAMENTO REGRESSIVO
#
# Porta fiel de runJpAlgorithmLogic() do backend TypeScript.
# Implementa Dijkstra sobre o grafo topológico de Wasserstein.
# O planejamento é "regressivo": define o objetivo X_objetivo primeiro,
# depois calcula os meios necessários até a ação imediata do usuário.
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

import heapq
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple


@dataclass
class PassoPlano:
    """
    Representa um passo no plano de comunicação gerado pelo Algoritmo JP.

    numero_passo        : posição na sequência (1-indexed)
    de_pictograma       : rótulo do pictograma de origem
    para_pictograma     : rótulo do pictograma de destino
    distancia_wasserstein: custo topológico da transição
    distancia_topologica: distância normalizada até o objetivo
    """
    numero_passo: int
    de_pictograma: str
    para_pictograma: str
    distancia_wasserstein: float
    distancia_topologica: float


@dataclass
class ResultadoAlgoritmoJP:
    """
    Resultado completo do planejamento regressivo pelo Algoritmo JP.
    """
    sucesso: bool
    caminho: List[PassoPlano]
    custo_total: float
    iteracoes: int
    mensagem: str
    vizinhos_imediatos: List[Dict]  # Top-3 próximos passos recomendados


class AlgoritmoJP:
    """
    Motor de Planejamento Regressivo da Inteligência Artificial Pictórica.

    Constrói um grafo topológico de pictogramas onde o peso de cada aresta
    é a Distância de Wasserstein entre os Diagramas de Persistência dos
    conceitos conectados.

    A equação G ≈ I + F é operacionalizada aqui:
      G (dinâmica)   = o caminho planejado pelo algoritmo
      I (incerteza)  = o estado atual do usuário (pictogramas selecionados)
      F (flexibilidade) = os 3 próximos passos de menor custo topológico
    """

    def __init__(self, pictogramas: List[Dict]):
        """
        Args:
            pictogramas: Lista de dicts com {id, palavra, categoria}
        """
        self.pictogramas = pictogramas
        self.idx_por_id = {p['id']: i for i, p in enumerate(pictogramas)}
        self.idx_por_palavra = {
            p['palavra'].lower(): i for i, p in enumerate(pictogramas)
        }
        print(f"   🔗 Grafo inicializado: {len(pictogramas)} nós")

        # Calcula matriz de distâncias Wasserstein
        self.dist_matrix = distancias_pairwise(pictogramas)
        print(f"   📐 Matriz de distâncias: {self.dist_matrix.shape}")

        # Calcula coordenadas 2D para visualização
        self.coords = mds_classico(self.dist_matrix)
        print(f"   🗺️  Coordenadas 2D calculadas via MDS Clássico")

    def _resolver_pictograma(self, id_ou_palavra: str) -> Optional[int]:
        """Resolve um ID ou palavra para o índice interno da lista."""
        try:
            pid = int(id_ou_palavra)
            return self.idx_por_id.get(pid)
        except ValueError:
            return self.idx_por_palavra.get(id_ou_palavra.lower())

    def vizinhos_topologicos(self, indice: int, top_k: int = 3) -> List[Dict]:
        """
        Retorna os top_k pictogramas mais próximos no espaço de Wasserstein.

        Esta é a operação central da "Flexibilidade" (F) na equação G ≈ I + F:
        dado onde o usuário está, quais são os próximos passos mais prováveis?

        Args:
            indice: Índice do pictograma atual na lista
            top_k:  Número de vizinhos a retornar (default 3)

        Returns:
            Lista de dicts com {palavra, categoria, distancia, coord_x, coord_y}
        """
        n = len(self.pictogramas)
        pares = [
            (self.dist_matrix[indice, j], j)
            for j in range(n) if j != indice
        ]
        pares.sort()

        resultado = []
        for dist, j in pares[:top_k]:
            p = self.pictogramas[j]
            resultado.append({
                'palavra': p['palavra'],
                'categoria': p['categoria'],
                'distancia_wasserstein': round(float(dist), 4),
                'coord_x': round(float(self.coords[j, 0]), 3),
                'coord_y': round(float(self.coords[j, 1]), 3),
            })

        return resultado

    def planejar(
        self,
        estado_atual: List[str],
        objetivo: str
    ) -> ResultadoAlgoritmoJP:
        """
        Executa o planejamento regressivo do Algoritmo JP.

        Dado um conjunto de pictogramas que representam o Estado Atual do
        usuário (sua incerteza I), e um pictograma objetivo X_objetivo,
        o algoritmo traça o caminho de menor custo topológico usando
        Dijkstra sobre o grafo de Wasserstein.

        Args:
            estado_atual: Lista de IDs ou palavras dos pictogramas selecionados
            objetivo:     ID ou palavra do pictograma alvo

        Returns:
            ResultadoAlgoritmoJP com caminho, custo e vizinhos imediatos
        """
        n = len(self.pictogramas)

        # Resolve índices
        idx_inicio = None
        for est in estado_atual:
            idx = self._resolver_pictograma(str(est))
            if idx is not None:
                idx_inicio = idx
                break  # Usa o primeiro pictograma válido como ponto de partida

        idx_objetivo = self._resolver_pictograma(str(objetivo))

        if idx_inicio is None or idx_objetivo is None:
            return ResultadoAlgoritmoJP(
                sucesso=False, caminho=[], custo_total=0.0, iteracoes=0,
                mensagem=f"Pictograma não encontrado: estado_atual={estado_atual}, objetivo={objetivo}",
                vizinhos_imediatos=[]
            )

        if idx_inicio == idx_objetivo:
            vizinhos = self.vizinhos_topologicos(idx_inicio)
            return ResultadoAlgoritmoJP(
                sucesso=True, caminho=[], custo_total=0.0, iteracoes=1,
                mensagem="Estado atual já é o objetivo.",
                vizinhos_imediatos=vizinhos
            )

        # ── Dijkstra sobre o grafo completo de Wasserstein ───────────────────
        # O grafo é completo: cada par de pictogramas está conectado
        # com custo = distância de Wasserstein normalizada

        distancias = [float('inf')] * n
        distancias[idx_inicio] = 0.0
        predecessores = [None] * n

        # Fila de prioridade: (custo_acumulado, índice)
        heap = [(0.0, idx_inicio)]
        visitados = set()
        iteracoes = 0

        while heap:
            iteracoes += 1
            custo_atual, u = heapq.heappop(heap)

            if u in visitados:
                continue
            visitados.add(u)

            if u == idx_objetivo:
                break

            # Explora todos os vizinhos (grafo completo)
            for v in range(n):
                if v == u or v in visitados:
                    continue
                custo_aresta = float(self.dist_matrix[u, v])
                novo_custo = custo_atual + custo_aresta

                if novo_custo < distancias[v]:
                    distancias[v] = novo_custo
                    predecessores[v] = u
                    heapq.heappush(heap, (novo_custo, v))

        # ── Reconstrução do caminho ──────────────────────────────────────────
        if distancias[idx_objetivo] == float('inf'):
            return ResultadoAlgoritmoJP(
                sucesso=False, caminho=[], custo_total=0.0, iteracoes=iteracoes,
                mensagem="Nenhum caminho encontrado.",
                vizinhos_imediatos=self.vizinhos_topologicos(idx_inicio)
            )

        # Reconstrói o caminho de trás para frente (planejamento regressivo)
        caminho_reverso = []
        atual = idx_objetivo
        while atual is not None and atual != idx_inicio:
            prev = predecessores[atual]
            if prev is None:
                break
            caminho_reverso.append((prev, atual))
            atual = prev

        caminho_reverso.reverse()  # Ordem cronológica: início → objetivo

        # Distância topológica normalizada até o objetivo
        dist_topo_total = float(self.dist_matrix[idx_inicio, idx_objetivo])

        passos = []
        for num, (de, para) in enumerate(caminho_reverso, start=1):
            n_passos = max(1, len(caminho_reverso))
            dist_para_obj = float(self.dist_matrix[para, idx_objetivo])
            passos.append(PassoPlano(
                numero_passo=num,
                de_pictograma=self.pictogramas[de]['palavra'],
                para_pictograma=self.pictogramas[para]['palavra'],
                distancia_wasserstein=round(float(self.dist_matrix[de, para]), 4),
                distancia_topologica=round(dist_para_obj, 4)
            ))

        vizinhos = self.vizinhos_topologicos(idx_inicio, top_k=3)

        p_inicio = self.pictogramas[idx_inicio]['palavra']
        p_obj    = self.pictogramas[idx_objetivo]['palavra']
        custo    = round(distancias[idx_objetivo], 4)

        return ResultadoAlgoritmoJP(
            sucesso=True,
            caminho=passos,
            custo_total=custo,
            iteracoes=iteracoes,
            mensagem=(
                f"Plano encontrado: {len(passos)} passo(s) de \"{p_inicio}\" "
                f"a \"{p_obj}\" com custo total {custo}."
            ),
            vizinhos_imediatos=vizinhos
        )


# ─── Preparação do conjunto de pictogramas IAP ───────────────────────────────
# Lista representativa de pictogramas ARASAAC para a simulação
# IDs baseados na base real ARASAAC (pt-BR)

PICTOGRAMAS_IAP = [
    # Necessidades
    {'id': 1, 'palavra': 'água',     'categoria': 'necessidades'},
    {'id': 2, 'palavra': 'comida',   'categoria': 'necessidades'},
    {'id': 3, 'palavra': 'fome',     'categoria': 'necessidades'},
    {'id': 4, 'palavra': 'sede',     'categoria': 'necessidades'},
    {'id': 5, 'palavra': 'dor',      'categoria': 'necessidades'},
    {'id': 6, 'palavra': 'remédio',  'categoria': 'necessidades'},
    {'id': 7, 'palavra': 'ajuda',    'categoria': 'necessidades'},
    {'id': 8, 'palavra': 'banheiro', 'categoria': 'necessidades'},
    # Sentimentos
    {'id': 11, 'palavra': 'feliz',   'categoria': 'sentimentos'},
    {'id': 12, 'palavra': 'triste',  'categoria': 'sentimentos'},
    {'id': 13, 'palavra': 'medo',    'categoria': 'sentimentos'},
    {'id': 14, 'palavra': 'cansado', 'categoria': 'sentimentos'},
    # Lugares
    {'id': 21, 'palavra': 'casa',    'categoria': 'lugares'},
    {'id': 22, 'palavra': 'hospital','categoria': 'lugares'},
    {'id': 23, 'palavra': 'escola',  'categoria': 'lugares'},
    # Pessoas
    {'id': 31, 'palavra': 'eu',      'categoria': 'pessoas'},
    {'id': 32, 'palavra': 'família', 'categoria': 'pessoas'},
    {'id': 33, 'palavra': 'médico',  'categoria': 'pessoas'},
    {'id': 34, 'palavra': 'amigo',   'categoria': 'pessoas'},
    # Ações
    {'id': 41, 'palavra': 'quero',   'categoria': 'acoes'},
    {'id': 42, 'palavra': 'sim',     'categoria': 'acoes'},
    {'id': 43, 'palavra': 'não',     'categoria': 'acoes'},
    {'id': 44, 'palavra': 'comer',   'categoria': 'acoes'},
    {'id': 45, 'palavra': 'ir',      'categoria': 'acoes'},
    {'id': 46, 'palavra': 'maçã',    'categoria': 'acoes'},
]

print("🗺️  Inicializando Algoritmo JP...")
jp = AlgoritmoJP(PICTOGRAMAS_IAP)
print()
print("✅ Algoritmo JP pronto!")

---
## 🎬 Célula 6 — Simulação da Prova de Conceito

### Cenário: Paciente com Afasia quer comunicar "Fome → Comer → Maçã"

Este é o momento em que a **"sustentação aerodinâmica"** fica visível:

1. O paciente seleciona o pictograma **"Fome"** (seu Estado Atual)
2. O Algoritmo JP traça o caminho até **"Maçã"** (o Estado Objetivo)
3. O grafo visualiza a rota, colorida por categoria semântica
4. Os 3 **vizinhos topológicos** de "Fome" são sugeridos como próximos passos

In [ ]:
# =============================================================================
# CÉLULA 6 — SIMULAÇÃO DA PROVA DE CONCEITO
#
# Cenário: Paciente com afasia quer comunicar "Fome → Comer → Maçã"
# Demonstra o Algoritmo JP em ação sobre o espaço topológico dos pictogramas.
#
# Output:
#   1. Resultado estruturado do planejamento (texto)
#   2. Grafo NetworkX com nós coloridos por categoria semântica
#   3. Atlas 2D com posições MDS e arestas do caminho ótimo
# =============================================================================

print("=" * 65)
print("  AFASIA — IAP: Simulação da Prova de Conceito")
print("  Algoritmo JP — Planejamento Regressivo")
print("=" * 65)
print()

# ─── Execução do Algoritmo JP ────────────────────────────────────────────────
ESTADO_ATUAL = ['fome']     # O que o paciente selecionou
OBJETIVO     = 'maçã'       # O que o paciente quer comunicar

print(f"📍 Estado Atual  : {ESTADO_ATUAL}")
print(f"🎯 Estado Objetivo: {OBJETIVO!r}")
print()

resultado = jp.planejar(estado_atual=ESTADO_ATUAL, objetivo=OBJETIVO)

# ─── Resultado Estruturado ───────────────────────────────────────────────────
print("─" * 65)
print(f"  {'✅' if resultado.sucesso else '❌'} {resultado.mensagem}")
print(f"  Custo Total (Wasserstein): {resultado.custo_total:.4f}")
print(f"  Iterações Dijkstra       : {resultado.iteracoes}")
print("─" * 65)
print()

if resultado.caminho:
    print("  📋 PLANO DE COMUNICAÇÃO:")
    print()
    for passo in resultado.caminho:
        barra = '█' * max(1, int(passo.distancia_wasserstein * 20))
        print(f"  Passo {passo.numero_passo}: {passo.de_pictograma!r:15s} "
              f"──[d={passo.distancia_wasserstein:.4f}]──► {passo.para_pictograma!r}")
        print(f"          Distância topo. até obj.: {passo.distancia_topologica:.4f}  {barra}")
    print()

print("  🔮 FLEXIBILIDADE (F) — Próximos 3 passos sugeridos:")
print()
for i, viz in enumerate(resultado.vizinhos_imediatos, 1):
    print(f"  {i}. {viz['palavra']:15s} [{viz['categoria']:12s}]  "
          f"d_Wasserstein = {viz['distancia_wasserstein']:.4f}  "
          f"coords=({viz['coord_x']:.3f}, {viz['coord_y']:.3f})")

print()
print("  G ≈ I + F:")
print(f"  G = caminho '{ESTADO_ATUAL[0]}' → '{OBJETIVO}'")
print(f"  I = incerteza do paciente (d total = {resultado.custo_total:.4f})")
print(f"  F = {len(resultado.vizinhos_imediatos)} próximos passos topológicos sugeridos")
print()


# ─── Visualização 1: Grafo de Dependências do Algoritmo JP ──────────────────
CORES_CATEGORIA = {
    'necessidades': '#f97316',   # Laranja
    'sentimentos':  '#a855f7',   # Roxo
    'lugares':      '#22c55e',   # Verde
    'pessoas':      '#3b82f6',   # Azul
    'acoes':        '#ef4444',   # Vermelho
    'outros':       '#6b7280',   # Cinza
}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0f172a')

# ── Painel 1: Grafo do caminho ótimo ────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#1e293b')
ax1.set_title('Algoritmo JP — Caminho Ótimo', color='white', fontsize=13, pad=12)

G_path = nx.DiGraph()

# Adiciona nós do caminho
nos_caminho = set()
for passo in resultado.caminho:
    nos_caminho.add(passo.de_pictograma)
    nos_caminho.add(passo.para_pictograma)
    G_path.add_edge(
        passo.de_pictograma,
        passo.para_pictograma,
        weight=passo.distancia_wasserstein,
        label=f"d={passo.distancia_wasserstein:.3f}"
    )

# Adiciona vizinhos topológicos ao grafo
inicio_label = ESTADO_ATUAL[0]
for viz in resultado.vizinhos_imediatos:
    G_path.add_edge(
        inicio_label,
        viz['palavra'],
        weight=viz['distancia_wasserstein'],
        label=f"d={viz['distancia_wasserstein']:.3f}",
        vizinho=True
    )
    nos_caminho.add(viz['palavra'])

# Cores dos nós
cat_por_palavra = {p['palavra']: p['categoria'] for p in PICTOGRAMAS_IAP}
node_colors = [CORES_CATEGORIA.get(cat_por_palavra.get(n, 'outros'), '#6b7280')
               for n in G_path.nodes()]

pos = nx.spring_layout(G_path, seed=42, k=2.0)

# Desenha arestas do caminho em branco; vizinhos em cinza tracejado
arestas_caminho = [(p.de_pictograma, p.para_pictograma) for p in resultado.caminho]
arestas_viz     = [(inicio_label, v['palavra']) for v in resultado.vizinhos_imediatos]

nx.draw_networkx_edges(G_path, pos, edgelist=arestas_caminho, ax=ax1,
                       edge_color='#22d3ee', arrows=True, arrowsize=25,
                       width=2.5, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_edges(G_path, pos, edgelist=arestas_viz, ax=ax1,
                       edge_color='#4b5563', arrows=True, arrowsize=15,
                       width=1.0, style='dashed', connectionstyle='arc3,rad=0.15')
nx.draw_networkx_nodes(G_path, pos, node_color=node_colors, node_size=800,
                       ax=ax1, alpha=0.95)
nx.draw_networkx_labels(G_path, pos, ax=ax1, font_color='white',
                        font_size=9, font_weight='bold')

# Labels nas arestas do caminho
edge_labels = {(p.de_pictograma, p.para_pictograma): f"d={p.distancia_wasserstein:.3f}"
               for p in resultado.caminho}
nx.draw_networkx_edge_labels(G_path, pos, edge_labels=edge_labels, ax=ax1,
                             font_color='#22d3ee', font_size=7.5)

# Legenda
patches = [mpatches.Patch(color=c, label=cat)
           for cat, c in CORES_CATEGORIA.items() if cat != 'outros']
ax1.legend(handles=patches, loc='upper right', fontsize=7.5,
           facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
ax1.axis('off')

# ── Painel 2: Atlas 2D com posições MDS ─────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#1e293b')
ax2.set_title('Atlas Topológico IAP — Espaço de Wasserstein (MDS 2D)',
              color='white', fontsize=13, pad=12)

coords = jp.coords  # shape (N, 2)
nomes_caminho_set = {p.de_pictograma for p in resultado.caminho} | \
                    {p.para_pictograma for p in resultado.caminho}

for i, picto in enumerate(PICTOGRAMAS_IAP):
    x, y = coords[i, 0], coords[i, 1]
    cor = CORES_CATEGORIA.get(picto['categoria'], '#6b7280')
    tamanho = 120 if picto['palavra'] in nomes_caminho_set else 60
    alpha   = 1.0 if picto['palavra'] in nomes_caminho_set else 0.55
    ax2.scatter(x, y, c=cor, s=tamanho, alpha=alpha, edgecolors='white',
                linewidths=0.8, zorder=3)
    ax2.annotate(
        picto['palavra'],
        (x, y), textcoords='offset points', xytext=(5, 4),
        fontsize=7.5, color='white',
        fontweight='bold' if picto['palavra'] in nomes_caminho_set else 'normal'
    )

# Desenha arestas do caminho sobre o atlas
palavra_para_idx = {p['palavra']: i for i, p in enumerate(PICTOGRAMAS_IAP)}
for passo in resultado.caminho:
    i_de   = palavra_para_idx.get(passo.de_pictograma)
    i_para = palavra_para_idx.get(passo.para_pictograma)
    if i_de is not None and i_para is not None:
        x0, y0 = coords[i_de,   0], coords[i_de,   1]
        x1, y1 = coords[i_para, 0], coords[i_para, 1]
        ax2.annotate('', xy=(x1, y1), xytext=(x0, y0),
                     arrowprops=dict(arrowstyle='->', color='#22d3ee',
                                     lw=2.0, mutation_scale=15))
        mid_x, mid_y = (x0 + x1) / 2, (y0 + y1) / 2
        ax2.annotate(f"d={passo.distancia_wasserstein:.3f}",
                     (mid_x, mid_y), fontsize=7, color='#22d3ee',
                     ha='center', zorder=5)

ax2.set_xlabel('Dimensão Topológica 1 (MDS)', color='#94a3b8', fontsize=9)
ax2.set_ylabel('Dimensão Topológica 2 (MDS)', color='#94a3b8', fontsize=9)
ax2.tick_params(colors='#64748b')
for spine in ax2.spines.values():
    spine.set_edgecolor('#334155')
ax2.legend(handles=patches, loc='upper right', fontsize=7.5,
           facecolor='#1e293b', edgecolor='#334155', labelcolor='white')

plt.suptitle(
    f'IAP — Inteligência Artificial Pictórica\n'
    f'Cenário: "{ESTADO_ATUAL[0]}" → "{OBJETIVO}"  |  '
    f'G ≈ I + F  |  d_Wasserstein total = {resultado.custo_total:.4f}',
    color='white', fontsize=11, y=1.01
)

plt.tight_layout()
plt.savefig('atlas_topologico_iap.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.show()

print()
print("✅ Visualização salva em: atlas_topologico_iap.png")
print()
print("=" * 65)
print("  🧠 A 'sustentação aerodinâmica' está provada:")
print("  O sistema navegou do espaço pré-linguístico de 'Fome'")
print(f"  até '{OBJETIVO}' em {len(resultado.caminho)} passo(s),")
print(f"  sem tokens, sem probabilidades, por geometria pura.")
print("="*65)